# RAG-Powered Q&A Pipeline
This notebook implements a Retrieval-Augmented Generation (RAG) pipeline using LangChain, OpenAI, and ChromaDB. 

**Objective:** Answer domain-specific questions based on a provided text document.

## Step 1: Install Dependencies
We need LangChain for orchestration, OpenAI for embeddings and LLM, and ChromaDB as our vector store.

In [ ]:
!pip install langchain langchain-openai chromadb python-dotenv

## Step 2: Environment Setup
Configure your OpenAI API key.

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()
os.environ["OPENAI_API_KEY"] = "your_openai_api_key_here" # Replace with your actual key

## Step 3: Document Loading and Splitting
Load the sample text and split it into manageable chunks. This ensures that the retrieved context fits within the LLM's context window and is semantically focused.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Load the local text file
loader = TextLoader("sample_data.txt")
documents = loader.load()

# Split the document into chunks
# chunk_overlap ensures that semantic context is preserved across splits
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
splits = text_splitter.split_documents(documents)
print(f"Created {len(splits)} chunks from the document.")

## Step 4: Vector Store and Embeddings
Convert the text chunks into embeddings using OpenAI and store them in ChromaDB for efficient similarity search.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize OpenAI Embeddings
embeddings = OpenAIEmbeddings()

# Store splits in ChromaDB
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
print("Vector database created and persisted.")

## Step 5: RAG Chain Setup
Create a retrieval chain that fetches relevant documents and passes them to the LLM with a custom prompt.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# Define a custom prompt to control the AI's behavior
template = """
You are a domain-specific expert. Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}
Question: {question}
Answer:"""

QA_CHAIN_PROMPT = PromptTemplate(input_variables=["context", "question"], template=template)

# Initialize the LLM
llm = ChatOpenAI(model_name="gpt-4-turbo", temperature=0)

# Create the RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(),
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
)
print("RAG Chain is ready.")

## Step 6: Testing the Pipeline
Ask a question based on the content of `sample_data.txt`.

In [ ]:
query = "What is Retrieval-Augmented Generation (RAG) and why is it useful?"
response = qa_chain.invoke(query)

print(f"Question: {query}")
print(f"Answer: {response['result']}")